# Submission

Training and validation run from `scripts/train.py`. Set a unique `EXPERIMENT_ID` for each model/submission so previous Kaggle uploads remain reproducible.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from joblib import load

EXPERIMENT_ID = 'catboost_balanced_001'  # set this to the model version you want to submit
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
model_path = ROOT / 'models' / f'{EXPERIMENT_ID}.joblib'
assert model_path.exists(), f'Model tidak ditemukan: {model_path}. Train model dengan ID yang sama terlebih dahulu.'
model = load(model_path)
test = pd.read_csv(ROOT / 'data' / 'test.csv')
sample = pd.read_csv(ROOT / 'data' / 'sample_submission.csv')
X_test = test.drop(columns=['building_id'])
raw_prediction = model.predict(X_test).ravel()
grade_prediction = pd.Series(raw_prediction).astype(str).str.extract(r'(\d)')[0].astype('int8')

submission = sample.copy()
assert submission['building_id'].equals(test['building_id']), 'Urutan building_id berbeda.'
assert len(grade_prediction) == len(submission), 'Jumlah prediksi tidak cocok.'
assert grade_prediction.between(1, 5).all(), 'damage_grade harus bernilai 1 sampai 5.'
submission['damage_grade'] = grade_prediction
output_path = ROOT / 'submissions' / f'{EXPERIMENT_ID}.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(output_path, index=False)
print(f'Saved {len(submission):,} predictions to {output_path}')
display(submission.head())